# Clase 150 — DPO / RLHF: alineamiento de LLMs

Implementamos **DPO loss desde scratch** con dataset sintético de preferencias. RLHF tradicional (SFT → RM → PPO) se explica conceptualmente.

In [ ]:
import numpy as np
rng = np.random.default_rng(42)

# Dataset sintético: 100 pares (prompt, chosen, rejected) como embeddings
n_pairs = 100
d_emb = 32

# Generamos "prompts" y respuestas: chosen siempre tiene un offset hacia un "good direction"
good_dir = rng.standard_normal(d_emb)
good_dir /= np.linalg.norm(good_dir)

prompts = rng.standard_normal((n_pairs, d_emb))
chosen = rng.standard_normal((n_pairs, d_emb)) + 1.5 * good_dir
rejected = rng.standard_normal((n_pairs, d_emb)) - 0.5 * good_dir

print(f'pairs: {n_pairs}, dim={d_emb}')
print(f'chosen·good = {(chosen @ good_dir).mean():.3f} (alto)')
print(f'rejected·good = {(rejected @ good_dir).mean():.3f} (bajo)')

## 1. Política simulada

Política `π(response|prompt)` = softmax sobre score lineal `θ^T · (prompt ⊕ response)`.

In [ ]:
def log_prob(theta, prompt, response):
    """log π(response|prompt) ≈ score lineal (log unnormalizado)."""
    feats = np.concatenate([prompt, response])
    return feats @ theta

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

# Política a entrenar y reference (frozen)
theta = rng.standard_normal(2 * d_emb) * 0.01
theta_ref = theta.copy()
beta = 0.3

## 2. DPO loss

$$ \mathcal{L}_{DPO} = -\log \sigma\!\left( \beta \cdot \left[\log\frac{\pi(y_w|x)}{\pi_{ref}(y_w|x)} - \log\frac{\pi(y_l|x)}{\pi_{ref}(y_l|x)}\right]\right) $$

In [ ]:
def dpo_loss_and_grad(theta, theta_ref, prompts, chosen, rejected, beta):
    losses = []
    grad = np.zeros_like(theta)
    for p, yw, yl in zip(prompts, chosen, rejected):
        fw = np.concatenate([p, yw]); fl = np.concatenate([p, yl])
        lpw, lpl = fw @ theta, fl @ theta
        lpw_r, lpl_r = fw @ theta_ref, fl @ theta_ref
        # logits diff
        h = beta * ((lpw - lpw_r) - (lpl - lpl_r))
        losses.append(-np.log(sigmoid(h) + 1e-9))
        # gradiente: dL/dtheta = -sigmoid(-h) * beta * (fw - fl)
        grad += -sigmoid(-h) * beta * (fw - fl)
    return np.mean(losses), grad / len(prompts)

# Accuracy: chosen tiene mayor log_prob que rejected
def pref_accuracy(theta, prompts, chosen, rejected):
    correct = 0
    for p, yw, yl in zip(prompts, chosen, rejected):
        if log_prob(theta, p, yw) > log_prob(theta, p, yl):
            correct += 1
    return correct / len(prompts)

print(f'accuracy inicial: {pref_accuracy(theta, prompts, chosen, rejected):.2%}')

## 3. Entrenamiento

In [ ]:
lr = 0.05
hist = []
for step in range(200):
    loss, g = dpo_loss_and_grad(theta, theta_ref, prompts, chosen, rejected, beta)
    theta -= lr * g
    if step % 20 == 0:
        acc = pref_accuracy(theta, prompts, chosen, rejected)
        hist.append((step, loss, acc))
        print(f'step {step:3d} | loss={loss:.4f} | acc={acc:.2%}')

print(f'\naccuracy final: {pref_accuracy(theta, prompts, chosen, rejected):.2%}')

## 4. Sensibilidad a β

- β alto → política se queda cerca del ref (regularización fuerte).
- β bajo → más agresivo, riesgo de overfit a preferencias.

In [ ]:
for beta_test in [0.05, 0.1, 0.3, 1.0]:
    th = theta_ref.copy()
    for _ in range(200):
        _, g = dpo_loss_and_grad(th, theta_ref, prompts, chosen, rejected, beta_test)
        th -= 0.05 * g
    drift = np.linalg.norm(th - theta_ref)
    acc = pref_accuracy(th, prompts, chosen, rejected)
    print(f'β={beta_test:4.2f} | KL drift ||θ-θ_ref||={drift:.3f} | acc={acc:.2%}')

## 5. RLHF clásico vs DPO

**RLHF (Ouyang 2022 / InstructGPT) — 3 etapas:**
1. **SFT**: fine-tune sobre (prompt, response_humana).
2. **Reward Model**: regresión sobre preferencias humanas con Bradley-Terry: `P(A≻B) = σ(r(A) − r(B))`.
3. **PPO**: optimizar el LLM contra el RM, con KL penalty al SFT.

Problemas: 4 modelos en memoria, training inestable, requiere infra grande.

**DPO (Rafailov 2023):**
- Derivación cerrada: la política óptima de RLHF se escribe como función del RM.
- Sustituyendo el RM en la pérdida de Bradley-Terry → eliminamos RM y PPO.
- Un solo step de fine-tuning supervisado sobre pares de preferencias.

**Variantes 2024:** IPO (identity link, evita overfit), KTO (sin pairs), ORPO (alineamiento desde SFT).

## 6. API real con TRL

```python
from trl import DPOTrainer, DPOConfig
from peft import LoraConfig
trainer = DPOTrainer(
    model=model, ref_model=ref_model,
    args=DPOConfig(beta=0.1, learning_rate=5e-5),
    train_dataset=ds, tokenizer=tok,
    peft_config=LoraConfig(r=16, lora_alpha=32))
trainer.train()
```

## Ejercicio guiado

1. Comparar DPO con ablación SIN ref model (= regresión logística sobre pares).
2. Implementar KTO loss (solo chosen o rejected, no pairs).
3. Métrica reward margin: `log π(chosen)/π_ref(chosen) − log π(rejected)/π_ref(rejected)`.

## Conclusiones

- DPO simplifica RLHF: 1 paso supervisado vs 3 etapas con PPO.
- β controla el trade-off ajuste vs regularización contra ref.
- Variantes (IPO, KTO, ORPO) cubren casos donde DPO falla.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios del README de esta clase. El código que usa librerías pesadas (`transformers` / `torch` / `keras` / `diffusers`) es la **API real** de la industria y se valida por sintaxis (los modelos requieren GPU/descarga). Los **núcleos numéricos** están en numpy puro, son **ejecutables** y se autoverifican con `assert`.

In [ ]:
try:
    import trl  # noqa: F401
    _HF = True
except Exception:
    _HF = False
print('trl disponible:', _HF)

### Ejercicio 1 — Dataset de preferencias `Anthropic/hh-rlhf`

In [ ]:
try:
    from datasets import load_dataset
    _DS = True
except Exception:
    _DS = False
if _DS:
    ds = load_dataset('Anthropic/hh-rlhf', split='train[:1%]')
    print(ds[0].keys())                     # 'chosen', 'rejected'
    print('chosen  :', ds[0]['chosen'][:80])
    print('rejected:', ds[0]['rejected'][:80])
else:
    print("load_dataset('Anthropic/hh-rlhf') -> columnas 'chosen' y 'rejected'"
          " (dialogos preferido vs no preferido).")

### Ejercicio 2 — DPO con TRL + LoRA (1 época)

In [ ]:
if _HF:
    from trl import DPOTrainer, DPOConfig
    from peft import LoraConfig
    trainer = DPOTrainer(
        model=model, ref_model=ref_model,
        args=DPOConfig(beta=0.1, learning_rate=5e-5, num_train_epochs=1),
        train_dataset=ds, tokenizer=tok,
        peft_config=LoraConfig(r=16, lora_alpha=32))
    trainer.train()
else:
    print('DPOTrainer(model, ref_model, DPOConfig(beta=0.1), peft_config=LoRA).train()')

### Ejercicio 3 — Eval pre/post: accuracy de preferencias sube (ejecutable)

In [ ]:
# Reutiliza dpo_loss_and_grad, pref_accuracy, theta_ref, prompts, chosen, rejected.
import numpy as np
acc_pre = pref_accuracy(theta_ref, prompts, chosen, rejected)
th = theta_ref.copy()
for _ in range(200):
    _, g = dpo_loss_and_grad(th, theta_ref, prompts, chosen, rejected, 0.3)
    th -= 0.05 * g
acc_post = pref_accuracy(th, prompts, chosen, rejected)
print(f'accuracy de preferencias  pre={acc_pre:.0%}  post={acc_post:.0%}')
assert acc_post > acc_pre        # DPO empuja chosen por encima de rejected

### Ejercicio 4 — Sensibilidad a β (ejecutable)

In [ ]:
import numpy as np
print('β    | KL drift ||θ-θ_ref|| | accuracy')
for beta in (0.1, 0.3, 1.0):
    th = theta_ref.copy()
    for _ in range(150):
        _, g = dpo_loss_and_grad(th, theta_ref, prompts, chosen, rejected, beta)
        th -= 0.05 * g
    drift = float(np.linalg.norm(th - theta_ref))
    acc = pref_accuracy(th, prompts, chosen, rejected)
    print(f'{beta:<4} | {drift:>18.3f} | {acc:.0%}')
    assert acc >= 0.9            # todas las β aprenden la preferencia
print('β regula cuanto se aleja la politica del modelo de referencia.')

### Ejercicio 5 — KTO: preferencias sin pares

In [ ]:
if _HF:
    from trl import KTOTrainer, KTOConfig
    # dataset con columnas: prompt, completion, label (True=deseable / False)
    trainer = KTOTrainer(model=model, ref_model=ref_model,
                         args=KTOConfig(beta=0.1), train_dataset=ds, tokenizer=tok)
    trainer.train()
else:
    print('KTO (Kahneman-Tversky): usa ejemplos sueltos etiquetados deseable/no'
          ' deseable, sin necesidad de pares (chosen, rejected).')